# Johansen Test on a Crypto Basket

This notebook applies the Johansen cointegration test to a basket of shared crypto close-price fixtures.

Chan's reason for using Johansen is practical: CADF is pairwise and order dependent, while Johansen can test a multivariate system and returns eigenvectors that can be used as hedge ratios.

The notebook reads local fixtures only. Refresh fixtures with:

```bash
python3 scripts/python/download-crypto-fixtures.py --source binance-monthly-archive
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import coint_johansen

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
fixture_path = Path("../../../../../fixtures/crypto/crypto_daily_close.csv")

if not fixture_path.exists():
    raise FileNotFoundError(
        "Missing shared crypto fixture: fixtures/crypto/crypto_daily_close.csv. "
        "Run `python3 scripts/python/download-crypto-fixtures.py --source binance-monthly-archive` "
        "from the repository root."
    )

close = pd.read_csv(fixture_path, parse_dates=["date"]).set_index("date").sort_index()
close = close.apply(pd.to_numeric, errors="coerce").dropna(how="all")

basket_symbols = ["BTCUSDT", "ETHUSDT", "BNBUSDT"]
missing = [symbol for symbol in basket_symbols if symbol not in close]
if missing:
    raise ValueError(f"Fixture is missing required symbols: {missing}")

prices = close[basket_symbols].dropna()
log_prices = np.log(prices)

log_prices.tail()

In [ ]:
rebased = log_prices - log_prices.iloc[0]
rebased.plot(figsize=(12, 5), title="Crypto basket log prices, rebased to zero")
plt.ylabel("Log price change from first fixture date")
plt.tight_layout();

In [ ]:
# det_order=0 allows a constant term in the cointegration relation.
# k_ar_diff=1 mirrors the one-lag setup used in Chan's examples.
johansen = coint_johansen(log_prices, det_order=0, k_ar_diff=1)

rank_table = pd.DataFrame(
    {
        "null_rank_at_most": range(len(basket_symbols)),
        "trace_stat": johansen.lr1,
        "trace_crit_90pct": johansen.cvt[:, 0],
        "trace_crit_95pct": johansen.cvt[:, 1],
        "trace_crit_99pct": johansen.cvt[:, 2],
        "max_eigen_stat": johansen.lr2,
        "max_eigen_crit_90pct": johansen.cvm[:, 0],
        "max_eigen_crit_95pct": johansen.cvm[:, 1],
        "max_eigen_crit_99pct": johansen.cvm[:, 2],
    }
)
rank_table["trace_reject_95pct"] = rank_table["trace_stat"] > rank_table["trace_crit_95pct"]
rank_table["max_eigen_reject_95pct"] = rank_table["max_eigen_stat"] > rank_table["max_eigen_crit_95pct"]

rank_table

In [ ]:
hedge_ratios = pd.DataFrame(
    johansen.evec,
    index=basket_symbols,
    columns=[f"relation_{i + 1}" for i in range(johansen.evec.shape[1])],
)

# Normalize each relation so BTC has unit weight. This changes scale, not the portfolio.
normalized_hedges = hedge_ratios.divide(hedge_ratios.loc["BTCUSDT"], axis=1)
normalized_hedges

In [ ]:
selected_relation = "relation_1"
weights = normalized_hedges[selected_relation]
portfolio = log_prices @ weights
portfolio.name = "cointegrating_portfolio"

portfolio.plot(figsize=(12, 5), title="Crypto cointegrating portfolio from Johansen relation 1")
plt.axhline(portfolio.mean(), color="black", linestyle="--", linewidth=1)
plt.ylabel("Weighted log-price portfolio")
plt.tight_layout();

weights.to_frame("portfolio_weight")

In [ ]:
def estimate_half_life(series: pd.Series) -> pd.Series:
    y = pd.Series(series).dropna()
    reg_df = pd.DataFrame({"lagged": y.shift(1), "delta": y.diff()}).dropna()
    design = np.column_stack([np.ones(len(reg_df)), reg_df["lagged"].to_numpy()])
    intercept, lambda_hat = np.linalg.lstsq(design, reg_df["delta"].to_numpy(), rcond=None)[0]

    half_life = np.nan if lambda_hat >= 0 else -np.log(2) / lambda_hat
    return pd.Series(
        {
            "intercept": intercept,
            "lambda": lambda_hat,
            "half_life_days": half_life,
            "observations": len(reg_df),
            "mean_reverting_lambda": lambda_hat < 0,
        }
    )

In [ ]:
half_life_summary = estimate_half_life(portfolio)
half_life_summary.to_frame("value")

## Interpretation

The rank table asks how many independent stationary relations survive the Johansen critical values. The eigenvectors then provide candidate hedge ratios. After selecting a relation, the resulting portfolio can be studied exactly like the spreads in Section 2.1: estimate pullback speed and translate it into a half-life.